# Administration - preamble
If you are seeing this section, then this notebook has been improperly rendered, and will thus be littered with code and other cells that are supposed to be hidden from general readers. *Keep this cell collapsed whenever possible.*

*Expand this cell to see legal notice and preamble. Click [here](#Administration---rendering) to go to the end of the document.*

## Disclaimer
The MIT License below applies to infrastructure provided by the notebook, and code snippets that may be used to generate supplementary academic content, except where otherwise noted.

The license does not automatically apply to academic content entered by the user, including, but not limited to, report text, answers, data, and figures. Consult your institution and/or your local jurisdiction for detailed advice.

## License
```
Copyright (c) 2026 John Isaac Calderon

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the “Software”), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED “AS IS”, WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
```

## Python definitions

In [1]:
import os
import sys
import shutil
import time

print(" I: Executing preamble code...", file=sys.stderr)

 I: Executing preamble code...


In [2]:
# Preamble is Copyright (c) 2026 John Isaac Calderon
# A copy of the MIT License is provided at the top of the notebook

# --- Preamble code START --- #

from IPython.display import display, Markdown, Latex
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import math
import time

# - Definitions - #

# verify project folder
cwd = os.path.realpath(".")
basename = os.path.basename(cwd)
assert len(cwd) > 0 and cwd != "/"

# Set the notebook name
docname = basename
nbname = f"main.ipynb"
pdfname = f"{docname}.pdf"

nbname_full = f"{cwd}/{nbname}"
pdfname_full = f"{cwd}/{pdfname}"


# Initialize temporary figures directory
figsdir = "tmpfigs"
figsdir_full = f"{cwd}/{figsdir}"

if os.path.exists(figsdir):
    shutil.rmtree(figsdir)

os.mkdir(figsdir)

print(f"  > Current directory: {cwd}", file=sys.stderr)
print(f"  > Target notebook: {nbname}", file=sys.stderr)
print(f"  > Output file: {pdfname}", file=sys.stderr)
print(f"  > Figures directory: {figsdir_full}", file=sys.stderr)

# Define a counter variable for the figures
FIG_CTR = 1


# - Functions - #


# Clear the page
def clearpage() -> None:
    display(Latex(r"\clearpage"))


# Display a generic image figure
def disp_imgfig(img: str, desc: str, preamble: str = "") -> None:
    global FIG_CTR

    # Make sure the file even exists
    if not os.path.exists(img):
        raise ValueError(f"Provided image file '{img}' does not exist.")

    # Create caption
    caption = f"**Figure {FIG_CTR}:** {desc}"

    # Create Markdown sequence
    md = f"|![{desc}]({img})|\n|:--:|\n|{caption}|"

    if preamble:
        md = preamble + "\n\n" + md

    # Display the sequence
    display(Markdown(md))

    # Increment the figure counter
    FIG_CTR += 1


# Display two generic image figures
def disp_imgfig2(
    img1: str, img2: str, desc1: str, desc2: str, preamble: str = ""
) -> None:
    global FIG_CTR

    # Make sure the files even exist
    if not os.path.exists(img1):
        raise ValueError(f"Provided image file '{img1}' does not exist.")

    if not os.path.exists(img2):
        raise ValueError(f"Provided image file '{img2}' does not exist.")

    # Create caption
    caption1 = f"**Figure {FIG_CTR}:** {desc1}"
    caption2 = f"**Figure {FIG_CTR+1}:** {desc2}"

    caption = caption1 + "\\\n" + caption2

    # Create Markdown sequence
    md = (
        f"|![{desc1}]({img1})|![{desc2}]({img2})|\n|:--:|:--:|\n|{caption1}|{caption2}|"
    )

    if preamble:
        md = preamble + "\n\n" + md

    # Display the sequence
    display(Markdown(md))

    # Increment the figure counter
    FIG_CTR += 2


# Display a Pyplot figure
def disp_pltfig(fig: plt.Figure, desc: str, preamble: str = "") -> None:
    global FIG_CTR

    # Save the figure as an image file, using the current
    # figure counter to discriminate between figures
    figfile = f"{figsdir}/tmpfig_{FIG_CTR}.png"

    # Save the figure and close it, so that
    # the notebook isn't littered with
    # orphaned figures
    fig.savefig(figfile, bbox_inches="tight", dpi=400)
    plt.close(fig)

    disp_imgfig(figfile, desc, preamble)


# Display two Pyplot figures
def disp_pltfig2(
    fig1: plt.Figure, fig2: plt.Figure, desc1: str, desc2: str, preamble: str = ""
) -> None:
    global FIG_CTR

    # Save the figure as an image file, using the current
    # figure counter to discriminate between figures
    figfile_1 = f"{figsdir}/tmpfig_{FIG_CTR}.png"
    figfile_2 = f"{figsdir}/tmpfig_{FIG_CTR+1}.png"

    # Save the figure and close it, so that
    # the notebook isn't littered with
    # orphaned figures
    fig1.savefig(figfile_1, bbox_inches="tight", dpi=400)
    fig2.savefig(figfile_2, bbox_inches="tight", dpi=400)
    plt.close(fig1)
    plt.close(fig2)

    disp_imgfig2(figfile_1, figfile_2, desc1, desc2, preamble)


# Find the `x` coordinates that satisfy `y1[...] == y2`
def where_is(
    x: np.ndarray, y1: np.ndarray, y2: float, err: float = 1.0e-3
) -> list[float]:
    assert len(x) == len(y1)

    a = []

    for u, v in zip(x, y1):
        if math.isclose(v, y2, rel_tol=err):
            a.append(u)

    return np.array(a)


# Display a table with labels
def disp_table(labels: list[str], *vals, backend="legacy") -> None:
    """
    Display a table with labels

    Parameters
    ----------
    labels: list[str]
        A list of labels

        The length (ie. the number of labels) defines the shape of the table.
    *vals
        Column vectors for each label ("column")

        The elements in the vectors should be representable (ie. applying
        `str(...)` to an element should be human-readable).

        The vectors should be of equal length, though this is not necessary,
        depending on the chosen backend

        Depending on the chosen backend, rounding
        may have to be performed by the caller.

    backend: str = 'pandas'
        Preferred data representation backend

        - 'pandas': Use `pandas.DataFrame` internally.
                    The value vectors must be of equal length.
                    WYSIWYG between notebook and PDF is not guaranteed.
        - 'legacy': Perform custom conditioning, then use Markdown internally.
                    The value vectors aren't required to be of equal length, as
                    shorter vectors will be padded internally.

    Returns
    -------
    This function does not return any values
    """

    if backend == "pandas":
        # - `pandas.DataFrame` expects a dictionary input
        data = dict(zip(labels, vals))
        data_pd = pd.DataFrame(data)
        display(data_pd)
    elif backend == "legacy":
        __disp_table(labels, *vals)


def __disp_table(labels: list[str], *vals) -> None:
    """
    Legacy Markdown-based backend for displaying a table with labels

    Parameters
    ----------
    labels: list[str]
        A list of labels

        The length (ie. the number of labels) defines the shape of the table.
    *vals
        Column vectors for each label ("column")

        The elements in the vectors should be representable (ie. applying
        `str(...)` to an element should be human-readable).

        The vectors should be of equal length, though this is not necessary,
        as shorter vectors will simply be padded by empty rows.

        Any rounding must be performed by the caller.

    Returns
    -------
    This function does not return any values
    """

    # Set up Markdown accumulator
    # - use the labels
    md = "|" + "|".join(labels) + "|"
    md += "\n"
    md += "|" + "|".join(":-:" for _ in labels) + "|"
    md += "\n"

    # Stringify the vectors, then extend the short ones
    # TODO: optimize this process
    v = [[str(j) for j in i] for i in vals]
    v_max = max(len(i) for i in vals)
    v = [i + ["--" for _ in range(v_max - len(i))] for i in v]

    # Generate the rest of the table
    for row in zip(*v):
        # - generate line
        line = "|" + "|".join(list(row)) + "|"

        # - append line to Markdown accumulator, then append newline
        md += line
        md += "\n"

    # Display resulting Markdown
    display(Markdown(md))


# --- Preamble code END --- #

  > Current directory: /home/johnc/Documents/Notebooks/template
  > Target notebook: main.ipynb
  > Output file: template.pdf
  > Figures directory: /home/johnc/Documents/Notebooks/template/tmpfigs


In [3]:
print(" I: Finished executing preamble code", file=sys.stderr)

 I: Finished executing preamble code


# Preface <a class="anchor" id="preface"></a>
We are not [there](#serendipity) yet.


# Serendipity
We are in fact there...

The value of $1 + 1$ is {{1 + 1}} (nah it's not gonna work)

# Administration - rendering
If you are seeing this section, then this notebook has been improperly rendered, and will thus be littered with code and other cells that are supposed to be hidden from general readers. *Keep this cell collapsed whenever possible.*

*Expand to show rendering procedure and build script.*

The code below renders this notebook into a PDF file. The notebook has a fixed name; the exported PDF file takes its name from the projet folder containing the notebook (basename).

## Rendering procedure
1. Click on this Markdown cell to bring it to focus.
    - In *Visual Studio Code*: Click on the Python cell below and select `Execute Above Cells`
2. Select `Kernel > Restart Kernel and Run up to Selected Cell`, then perform following sub-routine:

    - If execution stops abruptly, locate and correct any errors, then return to step 1
    - If execution completes without issues, commit changes made to the notebook: `Ctrl-S`

3. Click on the Python cell below and execute it: `Ctrl-Return`
4. Inspect the generated PDF document. Return to step 1 upon making any changes to the notebook.


In [5]:
# The question is: can a cell hide itself
print(f" I: Exporting notebook as PDF... (output file: {pdfname})", file=sys.stderr)

master = f"{cwd}/main.pdf"

t1 = time.time()

ecode = os.system(f'\
jupyter nbconvert \
--to pdf \
--TagRemovePreprocessor.remove_cell_tags="hide_cell" \
--TagRemovePreprocessor.remove_input_tags="hide_input" \
"{nbname}"')

t2 = time.time()

if ecode == 0:
    # move 'main.pdf' to '{docname}.pdf'
    if not os.path.exists(master):
        raise Exception(f"Could not find master export at 'main.pdf'...")
        # - we hope that Jupyter stops the execution right around here...

    shutil.move(master, pdfname_full)
    if not os.path.exists(pdfname_full):
        raise Exception(f"Could not find final export at '{pdfname}'")

    print(f" I: Export completed in {t2-t1:.3f} s.", file=sys.stderr)

# --- Nothing that is hidden, can be shown beyond this point --- #

 I: Exporting notebook as PDF... (output file: template.pdf)
[NbConvertApp] Converting notebook main.ipynb to pdf
[NbConvertApp] Writing 14621 bytes to main.pdf
 I: Export completed in 8.767 s.
